In [1]:
import torch
import torch.nn as nn
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset,DataLoader
from torchvision.models import vgg19,VGG19_Weights
import torch.optim as optim
import torchvision.transforms as T

In [2]:
device = torch.device('mps' if torch.mps.is_available else 'cpu')
model = vgg19(weights=VGG19_Weights.DEFAULT).to(device)
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

In [11]:
device

device(type='mps')

In [5]:
train_transform = T.Compose([
    T.Resize(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(),
    T.ToTensor(),
    T.Normalize(mean,std)
])
test_transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])
])
train = CIFAR10(root='./data',train=True,transform=train_transform)
test = CIFAR10(root='./data',train=False,transform=test_transform)

train_loader = DataLoader(dataset=train,batch_size=32,shuffle=True)
test_loader = DataLoader(dataset=test,batch_size=32,shuffle=False)

In [6]:
num_features = model.classifier[6].in_features
out_features = len(test.classes)
model.classifier[6] = nn.Linear(in_features=num_features,out_features=out_features)

In [7]:
for name, param in model.named_parameters():
    if not name.startswith('classifier'):
        param.requires_grad = False

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=1e-3)

In [14]:
model = model.to(device)
from tqdm import tqdm
def train_loop(model,train_loader,criterion,optimizer):
    model.train()
    run_loss, correct ,total = 0,0,0
    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits,labels)
        loss.backward()
        optimizer.step()
        preds = torch.argmax(logits,dim=1)
        run_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = run_loss/len(train_loader)
    epoch_acc = correct/total
    print(f'Epoch Loss : {epoch_loss:.4f} | Epoch Acc : {epoch_acc:.4f}')

In [15]:
for epoch in range(1):
    print(f'Epoch : {epoch+1}')
    train_loop(model=model,train_loader=train_loader,criterion=criterion,optimizer=optimizer)

Epoch : 1


 42%|████████████████▏                      | 650/1563 [06:35<09:16,  1.64it/s]


KeyboardInterrupt: 

In [16]:
@torch.no_grad()
def test_loop(model,test_loader,criterion,optimizer):
    model.eval()
    correct, total = 0,0
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        preds = torch.argmax(logits,dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Accuracy : {acc:.4f}")